# 1.Original model prediction

In [1]:
import sys
sys.path.append('./TITAN')
model_name='TITAN'
import argparse
import json
import logging
import os
import sys

import numpy as np
import torch
from paccmann_predictor.models import MODEL_FACTORY
from paccmann_predictor.utils.utils import get_device
from pytoda.datasets import (
    DrugAffinityDataset, ProteinProteinInteractionDataset
)
from pytoda.proteins import ProteinFeatureLanguage, ProteinLanguage
from pytoda.smiles.smiles_language import SMILESTokenizer
from sklearn.metrics import (
    auc, average_precision_score, precision_recall_curve, roc_curve
)
import os
#os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(123456)

# setup logging
logging.basicConfig(stream=sys.stdout)



def original_model_predict(
    test_affinity_filepath, receptor_filepath, ligand_filepath, model_path,
    model_type, preData_ori, resultfile_path
):
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)

    # Process parameter file:
    params_filepath = os.path.join(model_path, 'model_params.json')
    params = {}
    with open(params_filepath) as fp:
        params.update(json.load(fp))

    

    # Load languages

    smiles_language = SMILESTokenizer.from_pretrained(model_path)
    smiles_language.set_encoding_transforms(
        randomize=None,
        add_start_and_stop=params.get('ligand_start_stop_token', True),
        padding=params.get('ligand_padding', True),
        padding_length=params.get('ligand_padding_length', True),
    )
    smiles_language.set_smiles_transforms(
        augment=False,
        canonical=params.get('smiles_canonical', False),
        kekulize=params.get('smiles_kekulize', False),
        all_bonds_explicit=params.get('smiles_bonds_explicit', False),
        all_hs_explicit=params.get('smiles_all_hs_explicit', False),
        remove_bonddir=params.get('smiles_remove_bonddir', False),
        remove_chirality=params.get('smiles_remove_chirality', False),
        selfies=params.get('selfies', False),
        sanitize=params.get('sanitize', False)
    )
    if params.get('receptor_embedding', 'learned') == 'predefined':
        protein_language = ProteinFeatureLanguage.load(
            os.path.join(model_path, 'protein_language.pkl')
        )
    else:
        protein_language = ProteinLanguage.load(
            os.path.join(model_path, 'protein_language.pkl')
        )

    # Prepare the dataset
    logger.info("Start data preprocessing...")

    # Check if ligand as SMILES or as aa
    ligand_name, ligand_extension = os.path.splitext(ligand_filepath)
    if ligand_extension == '.csv':
        logger.info(
            'ligand file has extension .csv \n'
            'Please make sure ligand is provided as amino acid sequence.'
        )
        test_dataset = ProteinProteinInteractionDataset(
            sequence_filepaths=[[ligand_filepath], [receptor_filepath]],
            entity_names=['ligand_name', 'sequence_id'],
            labels_filepath=test_affinity_filepath,
            annotations_column_names=['label'],
            protein_languages=protein_language,
            padding_lengths=[
                params.get('ligand_padding_length', None),
                params.get('receptor_padding_length', None)
            ],
            paddings=params.get('ligand_padding', True),
            add_start_and_stops=params.get('add_start_stop_token', True),
            augment_by_reverts=params.get('augment_test_data', False),
            randomizes=False,
            iterate_datasets=True
        )

        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=False,
            num_workers=params.get('num_workers', 0)
        )

    elif ligand_extension == '.smi':
        logger.info(
            'ligand file has extension .smi \n'
            'Please make sure ligand is provided as SMILES.'
        )

        test_dataset = DrugAffinityDataset(
            drug_affinity_filepath=test_affinity_filepath,
            smi_filepath=ligand_filepath,
            protein_filepath=receptor_filepath,
            smiles_language=smiles_language,
            protein_language=protein_language,
            smiles_padding=params.get('ligand_padding', True),
            smiles_padding_length=params.get('ligand_padding_length', None),
            smiles_add_start_and_stop=params.get(
                'ligand_add_start_stop', True
            ),
            smiles_augment=False,
            smiles_canonical=params.get('test_smiles_canonical', False),
            smiles_kekulize=params.get('smiles_kekulize', False),
            smiles_all_bonds_explicit=params.get(
                'smiles_bonds_explicit', False
            ),
            smiles_all_hs_explicit=params.get('smiles_all_hs_explicit', False),
            smiles_remove_bonddir=params.get('smiles_remove_bonddir', False),
            smiles_remove_chirality=params.get(
                'smiles_remove_chirality', False
            ),
            smiles_selfies=params.get('selfies', False),
            protein_amino_acid_dict=params.get(
                'protein_amino_acid_dict', 'iupac'
            ),
            protein_padding=params.get('receptor_padding', True),
            protein_padding_length=params.get('receptor_padding_length', None),
            protein_add_start_and_stop=params.get(
                'receptor_add_start_stop', True
            ),
            protein_augment_by_revert=False,
            drug_affinity_dtype=torch.float,
            backend='eager',
            iterate_dataset=True
        )
        logger.info(f'Test dataset has {len(test_dataset)} samples.')
        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=True,
            num_workers=params.get('num_workers', 0)
        )
        logger.info(
            f'ligand_vocabulary_size  {smiles_language.number_of_tokens} '
            f'receptor_vocabulary_size {protein_language.number_of_tokens}.'
        )

    else:
        raise ValueError(
            f"Choose ligand_filepath with extension .csv or .smi, \
        given was {ligand_extension}"
        )
    logger.info(f'Test dataset has {len(test_dataset)} samples.')

    model_fn = params.get('model_fn', model_type)
    model = MODEL_FACTORY[model_fn](params).to(device)
    model._associate_language(smiles_language)
    model._associate_language(protein_language)

    model_file = os.path.join(
        model_path, 'weights', 'best_ROC-AUC_bimodal_mca.pt'
    )

    logger.info(f'looking for model in {model_file}')

    if os.path.isfile(model_file):
        logger.info('Found existing model, restoring now...')
        model.load(model_file, map_location=device)

        logger.info(f'model loaded: {model_file}')

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f'Number of parameters: {num_params}')

    # Measure validation performance
    loss_validation = []
    model.eval()
    with torch.no_grad():
        test_loss = 0
        predictions = []
        labels = []
        for ind, (ligand, receptors, y) in enumerate(test_loader):
            torch.cuda.empty_cache()
            y_hat, pred_dict = model(ligand.to(device), receptors.to(device))
            predictions.append(y_hat)
            labels.append(y.clone())
            loss = model.loss(y_hat, y.to(device))
            test_loss += loss.item()

    predictions = torch.cat(predictions, dim=0).flatten().cpu().numpy()
    labels = torch.cat(labels, dim=0).flatten().cpu().numpy()
    loss_validation.append(test_loss / len(test_loader))
    
    data_te=pd.read_csv(preData_ori)
    data_te.rename(columns={'CDR3B':'tcr','Epitope':'ligand','Affinity':'y_true'},inplace=True)
    probability = data_te[['tcr', 'ligand', 'y_true']]
    probability['y_prob'] = predictions
    probability['y_pred'] = probability['y_prob'].apply(lambda x: 1 if x >= 0.5 else 0)
    probability.to_csv(f'{resultfile_path}probalility.csv', index=False)
    print("Saving predictions")

    test_loss = test_loss / len(test_loader)
    fpr, tpr, _ = roc_curve(labels, predictions)
    test_roc_auc = auc(fpr, tpr)

    # calculations for visualization plot
    precision, recall, _ = precision_recall_curve(labels, predictions)
    avg_precision = average_precision_score(labels, predictions)

    logger.info(
        f"\t **** TESTING **** loss: {test_loss:.5f}, "
        f"ROC-AUC: {test_roc_auc:.3f}, Average precision: {avg_precision:.3f}."
    )


In [4]:
import sys
sys.path.append('./TITAN')

import argparse
import json
import logging
import os
import sys
import pandas as pd

import numpy as np
import torch
from paccmann_predictor.models import MODEL_FACTORY
from paccmann_predictor.utils.utils import get_device
from pytoda.datasets import (
    DrugAffinityDataset, ProteinProteinInteractionDataset
)
from pytoda.proteins import ProteinFeatureLanguage, ProteinLanguage
from pytoda.smiles.smiles_language import SMILESTokenizer
from sklearn.metrics import (
    auc, average_precision_score, precision_recall_curve, roc_curve
)

torch.manual_seed(123456)

# setup logging
logging.basicConfig(stream=sys.stdout)



def original_model_predict(
    test_affinity_filepath, receptor_filepath, ligand_filepath, model_path,
    model_type, preData_ori, resultfile_path
):
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)

    # Process parameter file:
    params_filepath = os.path.join(model_path, 'model_params.json')
    params = {}
    with open(params_filepath) as fp:
        params.update(json.load(fp))

    device = get_device()

    # Load languages

    smiles_language = SMILESTokenizer.from_pretrained(model_path)
    smiles_language.set_encoding_transforms(
        randomize=None,
        add_start_and_stop=params.get('ligand_start_stop_token', True),
        padding=params.get('ligand_padding', True),
        padding_length=params.get('ligand_padding_length', True),
    )
    smiles_language.set_smiles_transforms(
        augment=False,
        canonical=params.get('smiles_canonical', False),
        kekulize=params.get('smiles_kekulize', False),
        all_bonds_explicit=params.get('smiles_bonds_explicit', False),
        all_hs_explicit=params.get('smiles_all_hs_explicit', False),
        remove_bonddir=params.get('smiles_remove_bonddir', False),
        remove_chirality=params.get('smiles_remove_chirality', False),
        selfies=params.get('selfies', False),
        sanitize=params.get('sanitize', False)
    )
    if params.get('receptor_embedding', 'learned') == 'predefined':
        protein_language = ProteinFeatureLanguage.load(
            os.path.join(model_path, 'protein_language.pkl')
        )
    else:
        protein_language = ProteinLanguage.load(
            os.path.join(model_path, 'protein_language.pkl')
        )

    # Prepare the dataset
    logger.info("Start data preprocessing...")

    # Check if ligand as SMILES or as aa
    ligand_name, ligand_extension = os.path.splitext(ligand_filepath)
    if ligand_extension == '.csv':
        logger.info(
            'ligand file has extension .csv \n'
            'Please make sure ligand is provided as amino acid sequence.'
        )
        test_dataset = ProteinProteinInteractionDataset(
            sequence_filepaths=[[ligand_filepath], [receptor_filepath]],
            entity_names=['ligand_name', 'sequence_id'],
            labels_filepath=test_affinity_filepath,
            annotations_column_names=['label'],
            protein_languages=protein_language,
            padding_lengths=[
                params.get('ligand_padding_length', None),
                params.get('receptor_padding_length', None)
            ],
            paddings=params.get('ligand_padding', True),
            add_start_and_stops=params.get('add_start_stop_token', True),
            augment_by_reverts=params.get('augment_test_data', False),
            randomizes=False,
            iterate_datasets=True
        )

        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=False,
            num_workers=params.get('num_workers', 0)
        )

    elif ligand_extension == '.smi':
        logger.info(
            'ligand file has extension .smi \n'
            'Please make sure ligand is provided as SMILES.'
        )

        test_dataset = DrugAffinityDataset(
            drug_affinity_filepath=test_affinity_filepath,
            smi_filepath=ligand_filepath,
            protein_filepath=receptor_filepath,
            smiles_language=smiles_language,
            protein_language=protein_language,
            smiles_padding=params.get('ligand_padding', True),
            smiles_padding_length=params.get('ligand_padding_length', None),
            smiles_add_start_and_stop=params.get(
                'ligand_add_start_stop', True
            ),
            smiles_augment=False,
            smiles_canonical=params.get('test_smiles_canonical', False),
            smiles_kekulize=params.get('smiles_kekulize', False),
            smiles_all_bonds_explicit=params.get(
                'smiles_bonds_explicit', False
            ),
            smiles_all_hs_explicit=params.get('smiles_all_hs_explicit', False),
            smiles_remove_bonddir=params.get('smiles_remove_bonddir', False),
            smiles_remove_chirality=params.get(
                'smiles_remove_chirality', False
            ),
            smiles_selfies=params.get('selfies', False),
            protein_amino_acid_dict=params.get(
                'protein_amino_acid_dict', 'iupac'
            ),
            protein_padding=params.get('receptor_padding', True),
            protein_padding_length=params.get('receptor_padding_length', None),
            protein_add_start_and_stop=params.get(
                'receptor_add_start_stop', True
            ),
            protein_augment_by_revert=False,
            drug_affinity_dtype=torch.float,
            backend='eager',
            iterate_dataset=True
        )
        logger.info(f'Test dataset has {len(test_dataset)} samples.')
        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=True,
            num_workers=params.get('num_workers', 0)
        )
        logger.info(
            f'ligand_vocabulary_size  {smiles_language.number_of_tokens} '
            f'receptor_vocabulary_size {protein_language.number_of_tokens}.'
        )

    else:
        raise ValueError(
            f"Choose ligand_filepath with extension .csv or .smi, \
        given was {ligand_extension}"
        )
    logger.info(f'Test dataset has {len(test_dataset)} samples.')

    model_fn = params.get('model_fn', model_type)
    model = MODEL_FACTORY[model_fn](params).to(device)
    model._associate_language(smiles_language)
    model._associate_language(protein_language)

    model_file = os.path.join(
        model_path, 'weights', 'best_ROC-AUC_bimodal_mca.pt'
    )

    logger.info(f'looking for model in {model_file}')

    if os.path.isfile(model_file):
        logger.info('Found existing model, restoring now...')
        model.load(model_file, map_location=device)

        logger.info(f'model loaded: {model_file}')

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f'Number of parameters: {num_params}')

    # Measure validation performance
    loss_validation = []
    model.eval()
    with torch.no_grad():
        test_loss = 0
        predictions = []
        labels = []
        for ind, (ligand, receptors, y) in enumerate(test_loader):
            torch.cuda.empty_cache()
            y_hat, pred_dict = model(ligand.to(device), receptors.to(device))
            predictions.append(y_hat)
            labels.append(y.clone())
            loss = model.loss(y_hat, y.to(device))
            test_loss += loss.item()

    predictions = torch.cat(predictions, dim=0).flatten().cpu().numpy()
    labels = torch.cat(labels, dim=0).flatten().cpu().numpy()
    loss_validation.append(test_loss / len(test_loader))
    
    data_te=pd.read_csv(preData_ori)
    data_te.rename(columns={'CDR3B':'tcr','Epitope':'ligand','Affinity':'y_true'},inplace=True)
    probability = data_te[['tcr', 'ligand', 'y_true']]
    probability['y_prob'] = predictions
    probability['y_pred'] = probability['y_prob'].apply(lambda x: 1 if x >= 0.5 else 0)
    probability.to_csv(f'{resultfile_path}probability.csv', index=False)
    print("Saving predictions")

    test_loss = test_loss / len(test_loader)
    fpr, tpr, _ = roc_curve(labels, predictions)
    test_roc_auc = auc(fpr, tpr)

    # calculations for visualization plot
    precision, recall, _ = precision_recall_curve(labels, predictions)
    avg_precision = average_precision_score(labels, predictions)

    logger.info(
        f"\t **** TESTING **** loss: {test_loss:.5f}, "
        f"ROC-AUC: {test_roc_auc:.3f}, Average precision: {avg_precision:.3f}."
    )


In [11]:
from pathlib import Path
import pandas as pd

def seq_to_id_fast(df_seq):
    df = df_seq.rename(columns={'CDR3B': 'tcr', 'Epitope': 'ligand', 'Affinity': 'label'})[['tcr','ligand','label']]
    df_tcr = pd.DataFrame({'tcr': pd.Series(df['tcr'].dropna().unique())}).reset_index().rename(columns={'index':'sequence_id'})
    df_epi = pd.DataFrame({'ligand': pd.Series(df['ligand'].dropna().unique())}).reset_index().rename(columns={'index':'ligand_name'})
    out = df.merge(df_tcr, on='tcr', how='left').merge(df_epi, on='ligand', how='left')
    if out['sequence_id'].isna().any() or out['ligand_name'].isna().any():
        missing_tcr = out.loc[out['sequence_id'].isna(), 'tcr'].nunique()
        missing_epi = out.loc[out['ligand_name'].isna(), 'ligand'].nunique()
    out['sequence_id'] = out['sequence_id'].astype('Int64')  
    out['ligand_name'] = out['ligand_name'].astype('Int64')
    return out[['sequence_id','ligand_name','label']], df_tcr, df_epi

def fix_preData(data_ori, data_new, tcrData, epiData):
    for p in [data_new, tcrData, epiData]:
        Path(p).parent.mkdir(parents=True, exist_ok=True)
    data = pd.read_csv(data_ori)
    data_id, df_tcr, df_epi = seq_to_id_fast(data)
    df_tcr = df_tcr.assign(merged=df_tcr['tcr'].fillna('') + '\t' + df_tcr['sequence_id'].astype(str))
    df_epi = df_epi.assign(merged=df_epi['ligand'].fillna('') + '\t' + df_epi['ligand_name'].astype(str))
    data_id.to_csv(data_new, index=False)
    df_tcr[['merged']].to_csv(tcrData, index=False, header=False)
    df_epi[['merged']].to_csv(epiData, index=False, header=False)

    print('done saving!')


In [12]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [13]:
import pandas as pd
trainfile_path ="../data/train.csv"
data_new="../data/train_data_new.csv"
tcrData="../data/tcrData.csv"
epiData="../data/epiData.csv"
fix_preData(trainfile_path,data_new,tcrData,epiData)

done saving!


In [ ]:
model_type='bimodal_mca'
data_path="../data/train.csv"
test_affinity_filepath="../data/train_data_new.csv"
tcrData="../data/tcrData.csv"
epiData="../data/epiData.csv"
result_path="../result_path/Retraining_model_prediction"
model_path="../Retraining_model/"
original_model_predict(test_affinity_filepath, tcrData, epiData, model_path, model_type, data_path, result_path)

# 2.Model retraining

In [1]:
model_name='TITAN'
import os
os.chdir('./TITAN/')
import pandas as pd

import sys
sys.path.append('./TITAN')

import sys
print(sys.path)
import argparse
import json
import logging
import os
import sys
from time import time

import numpy as np
import torch
from paccmann_predictor.models import MODEL_FACTORY
from paccmann_predictor.utils.hyperparams import OPTIMIZER_FACTORY
from paccmann_predictor.utils.utils import get_device
from pytoda.datasets import (
    DrugAffinityDataset, ProteinProteinInteractionDataset
)
from pytoda.proteins import ProteinFeatureLanguage, ProteinLanguage
from pytoda.smiles.smiles_language import SMILESTokenizer
from sklearn.metrics import (
    auc, average_precision_score, precision_recall_curve, roc_curve
)
from pytoda.smiles import metadata

torch.manual_seed(123456)

# setup logging
logging.basicConfig(stream=sys.stdout)


def retrain_and_save(
    train_affinity_filepath, test_affinity_filepath, receptor_filepath,
    ligand_filepath,  model_dir, params_filepath, model_type, testData_ori, resultfile_path
):

    logger = logging.getLogger(f'{model_dir}')
    logger.setLevel(logging.DEBUG)
    # Process parameter file:
    params = {}
    with open(params_filepath) as fp:
        params.update(json.load(fp))

    # Create model directory and dump files
    # model_dir = os.path.join(model_path, training_name)
    os.makedirs(os.path.join(model_dir, 'weights'), exist_ok=True)
    os.makedirs(os.path.join(model_dir, 'results'), exist_ok=True)
    with open(os.path.join(model_dir, 'model_params.json'), 'w') as fp:
        json.dump(params, fp, indent=4)

    device = get_device()
    # Load languages
    smiles_language_filepath = os.path.join(
        os.sep,
        *metadata.__file__.split(os.sep)[:-1], 'tokenizer'
    )
    smiles_language = SMILESTokenizer.from_pretrained(smiles_language_filepath)
    smiles_language.set_encoding_transforms(
        randomize=None,
        add_start_and_stop=params.get('ligand_start_stop_token', True),
        padding=params.get('ligand_padding', True),
        padding_length=params.get('ligand_padding_length', True),
    )
    smiles_language.set_smiles_transforms(
        augment=params.get('augment_smiles', False),
        canonical=params.get('smiles_canonical', False),
        kekulize=params.get('smiles_kekulize', False),
        all_bonds_explicit=params.get('smiles_bonds_explicit', False),
        all_hs_explicit=params.get('smiles_all_hs_explicit', False),
        remove_bonddir=params.get('smiles_remove_bonddir', False),
        remove_chirality=params.get('smiles_remove_chirality', False),
        selfies=params.get('selfies', False),
        sanitize=params.get('sanitize', False)
    )

    if params.get('receptor_embedding', 'learned') == 'predefined':
        protein_language = ProteinFeatureLanguage(
            features=params.get('predefined_embedding', 'blosum')
        )
    else:
        protein_language = ProteinLanguage()

    if params.get('ligand_embedding', 'learned') == 'one_hot':
        logger.warning(
            'ligand_embedding_size parameter in param file is ignored in '
            'one_hot embedding setting, ligand_vocabulary_size used instead.'
        )
    if params.get('receptor_embedding', 'learned') == 'one_hot':
        logger.warning(
            'receptor_embedding_size parameter in param file is ignored in '
            'one_hot embedding setting, receptor_vocabulary_size used instead.'
        )

    # Prepare the dataset
    logger.info("Start data preprocessing...")

    # Check if peptide as SMILES or as aa
    pepname, pep_extension = os.path.splitext(ligand_filepath)
    if pep_extension == '.csv':
        logger.info(
            'Ligand file has extension .csv \n'
            'Please make sure ligand is provided as amino acid sequence.'
        )
        # Assemble datasets
        train_dataset = ProteinProteinInteractionDataset(
            sequence_filepaths=[[ligand_filepath], [receptor_filepath]],
            entity_names=['ligand_name', 'sequence_id'],
            labels_filepath=train_affinity_filepath,
            annotations_column_names=['label'],
            protein_languages=protein_language,
            padding_lengths=[
                params.get('ligand_padding_length', None),
                params.get('receptor_padding_length', None)
            ],
            paddings=params.get('ligand_padding', True),
            add_start_and_stops=params.get('add_start_stop_token', True),
            augment_by_reverts=params.get('augment_protein', False),
            randomizes=params.get('randomize', False),
            iterate_datasets=True
        )
        
        train_loader = torch.utils.data.DataLoader(
            dataset=train_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=False,
            num_workers=params.get('num_workers', 0)
        )

        test_dataset = ProteinProteinInteractionDataset(
            sequence_filepaths=[[ligand_filepath], [receptor_filepath]],
            entity_names=['ligand_name', 'sequence_id'],
            labels_filepath=test_affinity_filepath,
            annotations_column_names=['label'],
            protein_languages=protein_language,
            padding_lengths=[
                params.get('ligand_padding_length', None),
                params.get('receptor_padding_length', None)
            ],
            paddings=params.get('ligand_padding', True),
            add_start_and_stops=params.get('add_start_stop_token', True),
            augment_by_reverts=params.get('augment_test_data', False),
            randomizes=False,
            iterate_datasets=True
        )

        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=False,
            num_workers=params.get('num_workers', 0)
        )
        params.update({
            'ligand_vocabulary_size': protein_language.number_of_tokens,
            'receptor_vocabulary_size': protein_language.number_of_tokens,
            'ligand_as': 'amino acids'
        })  # yapf: disable
        logger.info(
            f'ligand_vocabulary_size {protein_language.number_of_tokens}, '
            f'receptor_vocabulary_size {protein_language.number_of_tokens}'
        )
        logger.info(
            f'Training dataset has {len(train_dataset)} samples, test set has '
            f'{len(test_dataset)}.'
        )


    elif pep_extension == '.smi':
        logger.info(
            'Ligand file has extension .smi \n'
            'Please make sure ligand is provided as SMILES.'
        )
        train_dataset = DrugAffinityDataset(
            drug_affinity_filepath=train_affinity_filepath,
            smi_filepath=ligand_filepath,
            protein_filepath=receptor_filepath,
            smiles_language=smiles_language,
            protein_language=protein_language,
            smiles_padding=params.get('ligand_padding', True),
            smiles_padding_length=params.get('ligand_padding_length', None),
            smiles_add_start_and_stop=params.get(
                'ligand_add_start_stop', True
            ),
            smiles_augment=params.get('augment_smiles', False),
            smiles_canonical=params.get('smiles_canonical', False),
            smiles_kekulize=params.get('smiles_kekulize', False),
            smiles_all_bonds_explicit=params.get(
                'smiles_bonds_explicit', False
            ),
            smiles_all_hs_explicit=params.get('smiles_all_hs_explicit', False),
            smiles_remove_bonddir=params.get('smiles_remove_bonddir', False),
            smiles_remove_chirality=params.get(
                'smiles_remove_chirality', False
            ),
            smiles_selfies=params.get('selfies', False),
            protein_amino_acid_dict=params.get(
                'protein_amino_acid_dict', 'iupac'
            ),
            protein_padding=params.get('receptor_padding', True),
            protein_padding_length=params.get('receptor_padding_length', None),
            protein_add_start_and_stop=params.get(
                'receptor_add_start_stop', True
            ),
            protein_augment_by_revert=params.get('augment_protein', False),
            drug_affinity_dtype=torch.float,
            backend='eager',
            iterate_dataset=True
        )
        train_loader = torch.utils.data.DataLoader(
            dataset=train_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=False,
            num_workers=params.get('num_workers', 0)
        )

        test_dataset = DrugAffinityDataset(
            drug_affinity_filepath=test_affinity_filepath,
            smi_filepath=ligand_filepath,
            protein_filepath=receptor_filepath,
            smiles_language=smiles_language,
            protein_language=protein_language,
            smiles_padding=params.get('ligand_padding', True),
            smiles_padding_length=params.get('ligand_padding_length', None),
            smiles_add_start_and_stop=params.get(
                'ligand_add_start_stop', True
            ),
            smiles_augment=False,
            smiles_canonical=params.get('test_smiles_canonical', False),
            smiles_kekulize=params.get('smiles_kekulize', False),
            smiles_all_bonds_explicit=params.get(
                'smiles_bonds_explicit', False
            ),
            smiles_all_hs_explicit=params.get('smiles_all_hs_explicit', False),
            smiles_remove_bonddir=params.get('smiles_remove_bonddir', False),
            smiles_remove_chirality=params.get(
                'smiles_remove_chirality', False
            ),
            smiles_selfies=params.get('selfies', False),
            protein_amino_acid_dict=params.get(
                'protein_amino_acid_dict', 'iupac'
            ),
            protein_padding=params.get('receptor_padding', True),
            protein_padding_length=params.get('receptor_padding_length', None),
            protein_add_start_and_stop=params.get(
                'receptor_add_start_stop', True
            ),
            protein_augment_by_revert=False,
            drug_affinity_dtype=torch.float,
            backend='eager',
            iterate_dataset=True
        )
        logger.info(
            f'Training dataset has {len(train_dataset)} samples, test set has '
            f'{len(test_dataset)}.'
        )
        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=False,
            num_workers=params.get('num_workers', 0)
        )

        params.update({
            'ligand_vocabulary_size': smiles_language.number_of_tokens,
            'receptor_vocabulary_size': protein_language.number_of_tokens,
            'ligand_as': 'smiles'
        })  # yapf: disable
        logger.info(
            f'ligand_vocabulary_size {smiles_language.number_of_tokens}, '
            f'receptor_vocabulary_size {protein_language.number_of_tokens}'
        )

    else:
        raise ValueError(
            f"Choose pep_filepath with extension .csv or .smi, \
        given was {pep_extension}"
        )

    save_top_model = os.path.join(model_dir, 'weights/{}_{}_{}.pt')

    model_fn = params.get('model_fn', model_type)
    model = MODEL_FACTORY[model_fn](params).to(device)
    model._associate_language(smiles_language)
    model._associate_language(protein_language)

    smiles_language.save_pretrained(model_dir)
    protein_language.save(os.path.join(model_dir, 'protein_language.pkl'))

    # Define optimizer
    min_loss, max_roc_auc = 100, 0
    optimizer = (
        OPTIMIZER_FACTORY[params.get('optimizer', 'adam')](
            model.parameters(),
            lr=params.get('lr', 0.001),
            weight_decay=params.get('weight_decay', 0.001)
        )
    )
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    params.update({'number_of_parameters': num_params})
    logger.info(f'Number of parameters: {num_params}')
    logger.info(f'Model: {model}')
    # Overwrite params.json file with updated parameters.
    with open(os.path.join(model_dir, 'model_params.json'), 'w') as fp:
        json.dump(params, fp)

    # Start training
    logger.info('Training about to start...\n')
    t = time()
    loss_training = []
    loss_validation = []

    model.save(save_top_model.format('epoch', '0', model_fn))

    for epoch in range(params['epochs']):

        model.train()
        logger.info(f"== Epoch [{epoch}/{params['epochs']}] ==")
        train_loss = 0

        for ind, (ligand, receptors, y) in enumerate(train_loader):

            torch.cuda.empty_cache()
            if ind % 10 == 0:
                logger.info(f'Batch {ind}/{len(train_loader)}')
            y_hat, pred_dict = model(ligand.to(device), receptors.to(device))
            loss = model.loss(y_hat, y.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        logger.info(
            "\t **** TRAINING ****   "
            f"Epoch [{epoch + 1}/{params['epochs']}], "
            f"loss: {train_loss / len(train_loader):.5f}. "
            f"This took {time() - t:.1f} secs."
        )

        t = time()

        model.eval()
        with torch.no_grad():
            test_loss = 0
            predictions = []
            labels = []
            for ind, (ligand, receptors, y) in enumerate(test_loader):
                torch.cuda.empty_cache()
                y_hat, pred_dict = model(
                    ligand.to(device), receptors.to(device)
                )
                predictions.append(y_hat)
                labels.append(y.clone())
                loss = model.loss(y_hat, y.to(device))
                test_loss += loss.item()

        predictions = torch.cat(predictions, dim=0).flatten().cpu().numpy()
        labels = torch.cat(labels, dim=0).flatten().cpu().numpy()

        data_te=pd.read_csv(testData_ori)
        data_te.rename(columns={'Affinity':'y_true'},inplace=True)
        probability = data_te[['CDR3B', 'Epitope', 'y_true']]
        probability['y_prob'] = predictions
        probability['y_pred'] = probability['y_prob'].apply(lambda x: 1 if x >= 0.5 else 0)
        probability.to_csv(f'{resultfile_path}probability.csv', index=False)
        print("Saving predictions")
        
        loss_validation.append(test_loss / len(test_loader))
        loss_training.append(train_loss / len(train_loader))

        test_loss = test_loss / len(test_loader)
        fpr, tpr, _ = roc_curve(labels, predictions)
        test_roc_auc = auc(fpr, tpr)

        # calculations for visualization plot
        precision, recall, _ = precision_recall_curve(labels, predictions)
        avg_precision = average_precision_score(labels, predictions)

        logger.info(
            f"\t **** TESTING **** Epoch [{epoch + 1}/{params['epochs']}], "
            f"loss: {test_loss:.5f}, ROC-AUC: {test_roc_auc:.3f}, "
            f"Average precision: {avg_precision:.3f}."
        )

        def save(path, metric, typ, val=None):
            model.save(path.format(typ, metric, model_fn))
            info = {
                'best_roc_auc': str(max_roc_auc),
                'test_loss': str(min_loss)
            }
            with open(
                os.path.join(model_dir, 'results', metric + '.json'), 'w'
            ) as f:
                json.dump(info, f)
            np.save(
                os.path.join(model_dir, 'results', metric + '_preds.npy'),
                np.vstack([predictions, labels])
            )
            if typ == 'best':
                logger.info(
                    f'\t New best performance in "{metric}"'
                    f' with value : {val:.7f} in epoch: {epoch}'
                )

        if test_roc_auc > max_roc_auc:
            max_roc_auc = test_roc_auc
            save(save_top_model, 'ROC-AUC', 'best', max_roc_auc)
            ep_roc = epoch
            roc_auc_loss = test_loss
            roc_auc_pr = avg_precision

        if test_loss < min_loss:
            min_loss = test_loss
            save(save_top_model, 'loss', 'best', min_loss)
            ep_loss = epoch
            loss_roc_auc = test_roc_auc
        if (epoch + 1) % params.get('save_model', 100) == 0:
            save(save_top_model, 'epoch', str(epoch))
            
    ep_roc = locals().get('ep_roc', params['epochs'])
    roc_auc_loss = locals().get('roc_auc_loss', test_roc_auc)
    roc_auc_pr = locals().get('roc_auc_pr', avg_precision)


        
    logger.info(
        'Overall best performances are: \n \t'
        f'Loss = {min_loss:.4f} in epoch {ep_loss} '
        f'\t (ROC-AUC was {loss_roc_auc:4f}) \n \t'
        f'ROC-AUC = {max_roc_auc:.4f} in epoch {ep_roc} '
        f'\t (Loss was {roc_auc_loss:4f})'
    )
    save(save_top_model, 'training', 'done')
    logger.info('Done with training, models saved, shutting down.')

    np.save(
        os.path.join(model_dir, 'results', 'loss_training.npy'), loss_training
    )
    np.save(
        os.path.join(model_dir, 'results', 'loss_validation.npy'),
        loss_validation
    )

['/home/wangyuyan/anaconda3/envs/titan/lib/python39.zip', '/home/wangyuyan/anaconda3/envs/titan/lib/python3.9', '/home/wangyuyan/anaconda3/envs/titan/lib/python3.9/lib-dynload', '', '/home/wangyuyan/anaconda3/envs/titan/lib/python3.9/site-packages', '/home/wangyuyan/TCREpitope/TITAN']


In [14]:
def seq_to_id(df_seq, df_tcr, df_epi):
    df_id=df_seq[['tcr','ligand','label']]
    for i in range(len(df_tcr)):
        df_id.loc[df_id.tcr==df_tcr['tcr'][i],'tcr_id']=df_tcr['index'][i]
    for i in range(len(df_epi)):
        df_id.loc[df_id.ligand==df_epi['ligand'][i],'ligand_id']=df_epi['index'][i]
    df_id['tcr_id']=df_id['tcr_id'].astype(int)
    df_id['ligand_id']=df_id['ligand_id'].astype(int)
    df_id=df_id[['tcr_id','ligand_id','label']]
    df_id.rename(columns={'ligand_id':'ligand_name','tcr_id':'sequence_id'},inplace=True)
    return df_id

def fix_traintestData(train_ori,test_ori,trainData,testData,tcrData,epiData):
    data_tr=pd.read_csv(train_ori)
    data_te=pd.read_csv(test_ori)
    data=pd.concat([data_tr,data_te],axis=0)
    data_tr.rename(columns={'CDR3B':'tcr','Epitope':'ligand','Affinity':'label'},inplace=True)
    data_te.rename(columns={'CDR3B':'tcr','Epitope':'ligand','Affinity':'label'},inplace=True)
    data.rename(columns={'CDR3B':'tcr','Epitope':'ligand','Affinity':'label'},inplace=True)
    
    df_tcr=data[['tcr']]
    df_tcr.drop_duplicates(inplace=True)
    df_tcr.reset_index(drop=True,inplace=True)
    df_tcr.reset_index(drop=False,inplace=True)
    df_epi=data[['ligand']]
    df_epi.drop_duplicates(inplace=True)
    df_epi.reset_index(drop=True,inplace=True)
    df_epi.reset_index(drop=False,inplace=True)

    data_tr_id=seq_to_id(data_tr, df_tcr, df_epi)
    data_te_id=seq_to_id(data_te, df_tcr, df_epi)

    df_tcr['index']=df_tcr['index'].astype(str)
    df_epi['index']=df_epi['index'].astype(str)
    df_tcr['merged'] = df_tcr['tcr'].fillna('') + '\t' + df_tcr['index'].fillna('')
    df_epi['merged'] = df_epi['ligand'].fillna('') + '\t' + df_epi['index'].fillna('')

    df_tcr=df_tcr[['merged']]
    df_epi=df_epi[['merged']]
     
    data_tr_id.to_csv(trainData, index=False)
    data_te_id.to_csv(testData, index=False)
    df_tcr.to_csv(tcrData, index=False,header=False)
    df_epi.to_csv(epiData, index=False,header=False)

In [ ]:
trainfile_path ="../data/train.csv"
testfile_path="../data/test.csv"
path="../data/file_out/"
os.makedirs(path, exist_ok=True)  
trainfile_name=path+'train_titan.csv'
testfile_name=path+'test_titan.csv'
tcrData=path+'tcr_traintest.csv'
epiData=path+'epi_traintest.csv'
fix_traintestData(trainfile_path, testfile_path, trainfile_name, testfile_name, tcrData,epiData)

In [ ]:
model_type='bimodal_mca'
params_filepath='./params/params_training.json'
testfile_path="../data/test.csv"
save_model_path="../Retraining_model/"
result_path="../result_path/Retraining_model_prediction"            
train_affinity_filepath=path+'train_titan.csv'
test_affinity_filepath=path+'test_titan.csv'
tcrData=path+'tcr_traintest.csv'
epiData=path+'epi_traintest.csv'
retrain_and_save(
train_affinity_filepath, test_affinity_filepath,tcrData, epiData, save_model_path,params_filepath, model_type, testfile_path, result_path)

# 3. Retraining model prediction

In [10]:
import sys
sys.path.append('./TITAN')

import argparse
import json
import logging
import os
import sys
import pandas as pd

import numpy as np
import torch
from paccmann_predictor.models import MODEL_FACTORY
from paccmann_predictor.utils.utils import get_device
from pytoda.datasets import (
    DrugAffinityDataset, ProteinProteinInteractionDataset
)
from pytoda.proteins import ProteinFeatureLanguage, ProteinLanguage
from pytoda.smiles.smiles_language import SMILESTokenizer
from sklearn.metrics import (
    auc, average_precision_score, precision_recall_curve, roc_curve
)

torch.manual_seed(123456)

# setup logging
logging.basicConfig(stream=sys.stdout)



def pre_and_save(
    test_affinity_filepath, receptor_filepath, ligand_filepath, model_path,
    model_type, preData_ori, resultfile_path
):
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)

    # Process parameter file:
    params_filepath = os.path.join(model_path, 'model_params.json')
    params = {}
    with open(params_filepath) as fp:
        params.update(json.load(fp))

    device = get_device()

    # Load languages

    smiles_language = SMILESTokenizer.from_pretrained(model_path)
    smiles_language.set_encoding_transforms(
        randomize=None,
        add_start_and_stop=params.get('ligand_start_stop_token', True),
        padding=params.get('ligand_padding', True),
        padding_length=params.get('ligand_padding_length', True),
    )
    smiles_language.set_smiles_transforms(
        augment=False,
        canonical=params.get('smiles_canonical', False),
        kekulize=params.get('smiles_kekulize', False),
        all_bonds_explicit=params.get('smiles_bonds_explicit', False),
        all_hs_explicit=params.get('smiles_all_hs_explicit', False),
        remove_bonddir=params.get('smiles_remove_bonddir', False),
        remove_chirality=params.get('smiles_remove_chirality', False),
        selfies=params.get('selfies', False),
        sanitize=params.get('sanitize', False)
    )
    if params.get('receptor_embedding', 'learned') == 'predefined':
        protein_language = ProteinFeatureLanguage.load(
            os.path.join(model_path, 'protein_language.pkl')
        )
    else:
        protein_language = ProteinLanguage.load(
            os.path.join(model_path, 'protein_language.pkl')
        )

    # Prepare the dataset
    logger.info("Start data preprocessing...")

    # Check if ligand as SMILES or as aa
    ligand_name, ligand_extension = os.path.splitext(ligand_filepath)
    if ligand_extension == '.csv':
        logger.info(
            'ligand file has extension .csv \n'
            'Please make sure ligand is provided as amino acid sequence.'
        )
        test_dataset = ProteinProteinInteractionDataset(
            sequence_filepaths=[[ligand_filepath], [receptor_filepath]],
            entity_names=['ligand_name', 'sequence_id'],
            labels_filepath=test_affinity_filepath,
            annotations_column_names=['label'],
            protein_languages=protein_language,
            padding_lengths=[
                params.get('ligand_padding_length', None),
                params.get('receptor_padding_length', None)
            ],
            paddings=params.get('ligand_padding', True),
            add_start_and_stops=params.get('add_start_stop_token', True),
            augment_by_reverts=params.get('augment_test_data', False),
            randomizes=False,
            iterate_datasets=True
        )

        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=False,
            num_workers=params.get('num_workers', 0)
        )

    elif ligand_extension == '.smi':
        logger.info(
            'ligand file has extension .smi \n'
            'Please make sure ligand is provided as SMILES.'
        )

        test_dataset = DrugAffinityDataset(
            drug_affinity_filepath=test_affinity_filepath,
            smi_filepath=ligand_filepath,
            protein_filepath=receptor_filepath,
            smiles_language=smiles_language,
            protein_language=protein_language,
            smiles_padding=params.get('ligand_padding', True),
            smiles_padding_length=params.get('ligand_padding_length', None),
            smiles_add_start_and_stop=params.get(
                'ligand_add_start_stop', True
            ),
            smiles_augment=False,
            smiles_canonical=params.get('test_smiles_canonical', False),
            smiles_kekulize=params.get('smiles_kekulize', False),
            smiles_all_bonds_explicit=params.get(
                'smiles_bonds_explicit', False
            ),
            smiles_all_hs_explicit=params.get('smiles_all_hs_explicit', False),
            smiles_remove_bonddir=params.get('smiles_remove_bonddir', False),
            smiles_remove_chirality=params.get(
                'smiles_remove_chirality', False
            ),
            smiles_selfies=params.get('selfies', False),
            protein_amino_acid_dict=params.get(
                'protein_amino_acid_dict', 'iupac'
            ),
            protein_padding=params.get('receptor_padding', True),
            protein_padding_length=params.get('receptor_padding_length', None),
            protein_add_start_and_stop=params.get(
                'receptor_add_start_stop', True
            ),
            protein_augment_by_revert=False,
            drug_affinity_dtype=torch.float,
            backend='eager',
            iterate_dataset=True
        )
        logger.info(f'Test dataset has {len(test_dataset)} samples.')
        test_loader = torch.utils.data.DataLoader(
            dataset=test_dataset,
            batch_size=params['batch_size'],
            shuffle=False,
            drop_last=True,
            num_workers=params.get('num_workers', 0)
        )
        logger.info(
            f'ligand_vocabulary_size  {smiles_language.number_of_tokens} '
            f'receptor_vocabulary_size {protein_language.number_of_tokens}.'
        )

    else:
        raise ValueError(
            f"Choose ligand_filepath with extension .csv or .smi, \
        given was {ligand_extension}"
        )
    logger.info(f'Test dataset has {len(test_dataset)} samples.')

    model_fn = params.get('model_fn', model_type)
    model = MODEL_FACTORY[model_fn](params).to(device)
    model._associate_language(smiles_language)
    model._associate_language(protein_language)

    model_file = os.path.join(
        model_path, 'weights', 'best_ROC-AUC_bimodal_mca.pt'
    )

    logger.info(f'looking for model in {model_file}')

    if os.path.isfile(model_file):
        logger.info('Found existing model, restoring now...')
        model.load(model_file, map_location=device)

        logger.info(f'model loaded: {model_file}')

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f'Number of parameters: {num_params}')

    # Measure validation performance
    loss_validation = []
    model.eval()
    with torch.no_grad():
        test_loss = 0
        predictions = []
        labels = []
        for ind, (ligand, receptors, y) in enumerate(test_loader):
            torch.cuda.empty_cache()
            y_hat, pred_dict = model(ligand.to(device), receptors.to(device))
            predictions.append(y_hat)
            labels.append(y.clone())
            loss = model.loss(y_hat, y.to(device))
            test_loss += loss.item()

    predictions = torch.cat(predictions, dim=0).flatten().cpu().numpy()
    labels = torch.cat(labels, dim=0).flatten().cpu().numpy()
    loss_validation.append(test_loss / len(test_loader))
    
    data_te=pd.read_csv(preData_ori)
    data_te.rename(columns={'CDR3B':'tcr','Epitope':'ligand','Affinity':'y_true'},inplace=True)
    probability = data_te[['tcr', 'ligand', 'y_true']]
    probability['y_prob'] = predictions
    probability['y_pred'] = probability['y_prob'].apply(lambda x: 1 if x >= 0.5 else 0)
    probability.to_csv(f'{resultfile_path}probability.csv', index=False)
    print("Saving predictions")

    test_loss = test_loss / len(test_loader)
    fpr, tpr, _ = roc_curve(labels, predictions)
    test_roc_auc = auc(fpr, tpr)

    # calculations for visualization plot
    precision, recall, _ = precision_recall_curve(labels, predictions)
    avg_precision = average_precision_score(labels, predictions)

    logger.info(
        f"\t **** TESTING **** loss: {test_loss:.5f}, "
        f"ROC-AUC: {test_roc_auc:.3f}, Average precision: {avg_precision:.3f}."
    )


In [ ]:
import pandas as pd
trainfile_path ="../data/validation.csv"
data_new=testfile_out+"../data/validation.csv"
tcrData=testfile_out+'tcr_validation.csv'
epiData=testfile_out+'epi_validation.csv'
fix_preData(trainfile_path,data_new,tcrData,epiData)

In [ ]:
testfile_path="../data/validation.csv"
save_model_path="../Retraining_model/"         
result_path=="../result_path/Retraining_model_prediction"            
testfile_out=="../data/file_out/"        
test_affinity_filepath=testfile_out+'validation.csv'
tcrData=testfile_out+'tcr_validation.csv'
epiData=testfile_out+'epi_validation.csv'
pre_and_save(test_affinity_filepath, tcrData,epiData, save_model_path, model_type, testfile_path, result_path)